# DoRA vs LoRA 对比实验（A100）

**目的**：将 LoRA 替换为 DoRA，其他参数完全不变，对比 GSM8K 效果差异。

**DoRA 原理**：将预训练权重分解为幅度（magnitude）和方向（direction），仅对方向做低秩适配，不增加推理开销但更接近全量微调效果。

**对比设计**：
- 数据集、超参数、模型、seed 完全一致
- 唯一变量：`use_dora=True`
- 输出目录隔离：`sft_dora` / `dpo_dora`

> 请确保已跑完 LoRA 基线（colab_train.ipynb），以便对比。

In [ ]:
# Cell 1：挂载 Drive + 设置 Token
import os
from google.colab import drive, userdata

drive.mount('/content/drive')
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
!mkdir -p "{DRIVE_DIR}/outputs"
print("✅ Drive 已挂载，HF Token 已加载")

In [ ]:
# Cell 2：克隆项目 + 安装依赖
import os

PROJECT_DIR = "/content/6000Q-QwenMiniReason"
REPO_URL = f"https://{os.environ['HF_TOKEN']}@github.com/yukiiii0730/6000Q-QwenMiniReason.git"
BRANCH = "Nancy"

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} fetch origin
    !git -C {PROJECT_DIR} reset --hard origin/{BRANCH}

%cd {PROJECT_DIR}
!pip install -r requirements.txt -q

import unsloth, trl, transformers
print(f"✅ Unsloth {unsloth.__version__} | TRL {trl.__version__} | Transformers {transformers.__version__}")

In [ ]:
# Cell 3：生成 DoRA 专用配置
#
# 与 LoRA 实验的唯一区别：
#   1. lora.use_dora = true
#   2. 输出目录改为 sft_dora / dpo_dora（与 LoRA 隔离）
#   3. 其他所有参数完全一致

import yaml

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"

# ── 模式切换 ───────────────────────────────────────────────────
FAST_TEST = True   # True=快速测试, False=正式训练

if FAST_TEST:
    SFT_MAX_STEPS, DPO_MAX_STEPS = 50, 30
    print("⚡ 快速测试模式")
else:
    SFT_MAX_STEPS, DPO_MAX_STEPS = 1800, 800
    print("🔥 正式训练模式")

# ── SFT DoRA 配置 ──────────────────────────────────────────────
with open("config/sft_config.yaml", "r") as f:
    sft_cfg = yaml.safe_load(f)

sft_cfg["output_dir"] = f"{DRIVE_DIR}/outputs/sft_dora"     # 隔离输出
sft_cfg["load_in_4bit"] = True
sft_cfg["lora"]["use_dora"] = True                           # ← 唯一变量
sft_cfg["train"]["per_device_train_batch_size"] = 16
sft_cfg["train"]["gradient_accumulation_steps"] = 1
sft_cfg["train"]["max_steps"] = SFT_MAX_STEPS
sft_cfg["train"]["bf16"] = True
sft_cfg["train"]["fp16"] = False
sft_cfg.pop("datasets", None)

with open("config/sft_config.yaml", "w") as f:
    yaml.dump(sft_cfg, f, allow_unicode=True)

# ── DPO DoRA 配置 ──────────────────────────────────────────────
with open("config/dpo_config.yaml", "r") as f:
    dpo_cfg = yaml.safe_load(f)

dpo_cfg["output_dir"] = f"{DRIVE_DIR}/outputs/dpo_dora"     # 隔离输出
dpo_cfg["base_adapter_path"] = f"{DRIVE_DIR}/outputs/sft_dora"
dpo_cfg["load_in_4bit"] = True
dpo_cfg["lora"]["use_dora"] = True                           # ← 唯一变量
dpo_cfg["train"]["per_device_train_batch_size"] = 8
dpo_cfg["train"]["gradient_accumulation_steps"] = 2
dpo_cfg["train"]["max_steps"] = DPO_MAX_STEPS
dpo_cfg["train"]["bf16"] = True
dpo_cfg["train"]["fp16"] = False
dpo_cfg.pop("dataset", None)

with open("config/dpo_config.yaml", "w") as f:
    yaml.dump(dpo_cfg, f, allow_unicode=True)

print(f"\n📋 DoRA 实验配置：")
print(f"   SFT  → {sft_cfg['output_dir']}  ({SFT_MAX_STEPS} steps)")
print(f"   DPO  → {dpo_cfg['output_dir']}  ({DPO_MAX_STEPS} steps)")
print(f"   use_dora = True")
print("✅ DoRA 配置已生成")

In [ ]:
# Cell 4：准备数据（复用 LoRA 实验的缓存，无需重新下载）
import json, os
from datasets import load_dataset

DRIVE_DIR  = "/content/drive/MyDrive/Qwen-Reasoning"
PROCESSED  = f"{DRIVE_DIR}/data/processed"
os.makedirs(PROCESSED, exist_ok=True)

SFT_OUT = f"{PROCESSED}/sft_train.json"
DPO_OUT = f"{PROCESSED}/dpo_train.json"

DEFAULT_INSTRUCTION = "请先进行清晰的逐步推理，再给出最终答案。"

if FAST_TEST:
    SFT_NUMINA_N, SFT_MAGPIE_N, DPO_ORCA_N = 500, 500, 500
else:
    SFT_NUMINA_N, SFT_MAGPIE_N, DPO_ORCA_N = 50000, 50000, 50000

# SFT 数据
if os.path.exists(SFT_OUT):
    print(f"⏭️  SFT 缓存已存在: {SFT_OUT}")
else:
    print(f"🔄 转换 SFT 数据集...")
    sft_rows = []
    numina = load_dataset("AI-MO/NuminaMath-CoT", split="train", trust_remote_code=True).select(range(SFT_NUMINA_N))
    for x in numina:
        sft_rows.append({"instruction": DEFAULT_INSTRUCTION, "input": x["problem"].strip(), "output": x["solution"].strip()})
    magpie = load_dataset("Magpie-Align/Magpie-Reasoning-150K", split="train", trust_remote_code=True).select(range(SFT_MAGPIE_N))
    for x in magpie:
        sft_rows.append({"instruction": x["instruction"].strip(), "input": "", "output": x["response"].strip()})
    with open(SFT_OUT, "w", encoding="utf-8") as f:
        json.dump(sft_rows, f, ensure_ascii=False, indent=2)
    print(f"✅ SFT: {len(sft_rows)} 条 → {SFT_OUT}")

# DPO 数据
if os.path.exists(DPO_OUT):
    print(f"⏭️  DPO 缓存已存在: {DPO_OUT}")
else:
    print(f"🔄 转换 DPO 数据集...")
    orca = load_dataset("microsoft/orca-math-word-problems-200k", split="train", trust_remote_code=True).select(range(DPO_ORCA_N))
    dpo_rows = []
    for x in orca:
        rejected = x.get("incorrect_solution", "")
        if not rejected: continue
        dpo_rows.append({"prompt": x.get("question","").strip(), "chosen": x.get("correct_solution","").strip(), "rejected": rejected.strip()})
    with open(DPO_OUT, "w", encoding="utf-8") as f:
        json.dump(dpo_rows, f, ensure_ascii=False, indent=2)
    print(f"✅ DPO: {len(dpo_rows)} 条 → {DPO_OUT}")

# 写回 config
import yaml
with open("config/sft_config.yaml", "r") as f:
    sft_cfg = yaml.safe_load(f)
sft_cfg["dataset_path"] = SFT_OUT
with open("config/sft_config.yaml", "w") as f:
    yaml.dump(sft_cfg, f, allow_unicode=True)

with open("config/dpo_config.yaml", "r") as f:
    dpo_cfg = yaml.safe_load(f)
dpo_cfg["dataset_path"] = DPO_OUT
with open("config/dpo_config.yaml", "w") as f:
    yaml.dump(dpo_cfg, f, allow_unicode=True)

print("✅ 数据准备完成，配置已更新")

In [ ]:
# Cell 5：DoRA SFT 训练
!python scripts/sft_train.py --config config/sft_config.yaml

In [ ]:
# Cell 6：验证 SFT DoRA 产物
import os
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
print("SFT DoRA 输出:", os.listdir(f"{DRIVE_DIR}/outputs/sft_dora"))

In [ ]:
# Cell 7：DoRA DPO 训练
!python scripts/dpo_train.py --config config/dpo_config.yaml

In [ ]:
# Cell 8：合并 DoRA 权重
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
!python scripts/merge_lora.py \
    --adapter_path "{DRIVE_DIR}/outputs/dpo_dora" \
    --output_path  "{DRIVE_DIR}/outputs/merged_dora"
print("✅ DoRA 权重合并完成")

In [ ]:
# Cell 9：GSM8K 评测（DoRA 模型）
DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"
GSM8K_N = 50 if FAST_TEST else 1319

!python eval/gsm8k_eval.py \
    --model_path "{DRIVE_DIR}/outputs/merged_dora" \
    --max_samples {GSM8K_N} \
    --output eval/gsm8k_dora_result.json

import json
with open("eval/gsm8k_dora_result.json") as f:
    dora_result = json.load(f)
print(f"\n✅ DoRA GSM8K: {dora_result['accuracy']:.2%}  ({dora_result['correct']}/{dora_result['total']})")

In [ ]:
# Cell 10：LoRA vs DoRA 对比
import json

DRIVE_DIR = "/content/drive/MyDrive/Qwen-Reasoning"

# 读取各组结果
results = {}
files = {
    "Baseline (原始模型)": "eval/gsm8k_baseline.json",
    "LoRA (SFT+DPO)":     "eval/gsm8k_result.json",
    "DoRA (SFT+DPO)":     "eval/gsm8k_dora_result.json",
}
for label, path in files.items():
    try:
        with open(path) as f:
            data = json.load(f)
        results[label] = data["accuracy"]
    except FileNotFoundError:
        results[label] = None

# 打印对比表
print("\n" + "=" * 50)
print(f"  {'模型':<25} {'GSM8K':>10}")
print("=" * 50)
baseline_acc = results.get("Baseline (原始模型)")
for label, acc in results.items():
    acc_str = f"{acc:.2%}" if acc is not None else "N/A"
    delta = ""
    if acc is not None and baseline_acc is not None and label != "Baseline (原始模型)":
        delta = f"  ({acc - baseline_acc:+.2%})"
    print(f"  {label:<25} {acc_str:>10}{delta}")
print("=" * 50)

# LoRA vs DoRA 直接对比
lora_acc = results.get("LoRA (SFT+DPO)")
dora_acc = results.get("DoRA (SFT+DPO)")
if lora_acc is not None and dora_acc is not None:
    diff = dora_acc - lora_acc
    winner = "DoRA" if diff > 0 else "LoRA" if diff < 0 else "持平"
    print(f"\n🏆 DoRA vs LoRA: {diff:+.2%}  → {winner} 更优")

# 保存对比结果
compare = {"results": results}
with open("eval/lora_vs_dora_compare.json", "w", encoding="utf-8") as f:
    json.dump(compare, f, ensure_ascii=False, indent=2)
print(f"\n📝 对比结果已保存: eval/lora_vs_dora_compare.json")

In [ ]:
# Cell 11（可选）：推送评测结果到 GitHub
!git add eval/gsm8k_dora_result.json eval/lora_vs_dora_compare.json
!git commit -m "Add DoRA vs LoRA comparison results" || true
!git push origin Nancy || true